# GDD_ENS membership inference (LiRA + RMIA + population)

Audits a single-MLP tumor-type classifier (38 tumor types) for membership leakage.

The population is built from the GDD **full feature table** `msk_solid_heme_ft.csv`: we keep
the `train`-category samples, take one sample per patient (so members and non-members never
share a patient), label-encode the target, drop identifier/metadata and constant feature
columns, and do **no** oversampling — LeakPro carves the in/out split from this natural
population.

**Prerequisite:** place `msk_solid_heme_ft.csv` in `./data/` first (see the README). It is not
shipped (GENIE is controlled-access; the file is ~860 MB).

Runs under python3.12 / the LeakPro `.venv`.

In [ ]:
import os
import sys
import yaml

project_root = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.insert(0, project_root)

## 1. Prepare the population

Read the full feature table, keep `train`-category samples, take one sample per patient,
label-encode `Cancer_Type`, drop metadata + constant feature columns, wrap in a `UserDataset`,
and cache it as a pickle for LeakPro.

In [ ]:
import pickle

from gdd_data_handler import GddDataHandler
from utils.gdd_data import prepare_gdd_population

with open('train_config.yaml', 'r') as file:
    train_config = yaml.safe_load(file)

data_dir = train_config['data']['data_dir']
dataset_name = train_config['data']['dataset']
ft_path = os.path.join(data_dir, train_config['data']['feature_table'])
population_path = os.path.join(data_dir, 'gdd_ens.pkl')

if os.path.exists(population_path):
    with open(population_path, 'rb') as file:
        population_dataset = pickle.load(file)
    print(f'Loaded cached population from {population_path}')
else:
    features, targets, label_encoder = prepare_gdd_population(ft_path)
    population_dataset = GddDataHandler.UserDataset(features, targets)
    with open(population_path, 'wb') as file:
        pickle.dump(population_dataset, file)
    print(f'Saved population ({len(population_dataset)} samples, '
          f'{features.shape[1]} features, {len(label_encoder.classes_)} classes) '
          f'to {population_path}')

print('Population size:', len(population_dataset))

## 2. Carve target train/test splits

`f_train` / `f_test` of the population train and evaluate the target; the remaining samples
form the shadow-model pool LeakPro draws from.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

np.random.seed(train_config['run']['random_seed'])

train_fraction = train_config['data']['f_train']
test_fraction = train_config['data']['f_test']
batch_size = train_config['train']['batch_size']

dataset_size = len(population_dataset)
train_size = int(train_fraction * dataset_size)
test_size = int(test_fraction * dataset_size)

selected_index = np.random.choice(np.arange(dataset_size), train_size + test_size, replace=False)
train_indices, test_indices = train_test_split(selected_index, test_size=test_size)

data = population_dataset.data
targets = population_dataset.targets
train_subset = GddDataHandler.UserDataset(data[train_indices], targets[train_indices])
test_subset = GddDataHandler.UserDataset(data[test_indices], targets[test_indices])

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)
print(f'Train: {len(train_subset)} | Test: {len(test_subset)} | '
      f'Shadow pool: {dataset_size - len(train_subset) - len(test_subset)}')

## 3. Train the target model and record its metadata

The recorded recipe (init params, optimizer, criterion, epochs) is exactly what LeakPro
replays to train shadow models.

In [ ]:
from torch import nn, optim, save

from gdd_model_handler import GddModelHandler
from utils.gdd_model import GddMLP
from leakpro import LeakPro

log_dir = train_config['run']['log_dir']
os.makedirs(log_dir, exist_ok=True)

lr = train_config['train']['learning_rate']
weight_decay = train_config['train']['weight_decay']
epochs = train_config['train']['epochs']

input_size = population_dataset.data.shape[1]
num_classes = int(population_dataset.targets.max().item()) + 1
model = GddMLP(input_size=input_size, num_classes=num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

train_result = GddModelHandler().train(
    dataloader=train_loader, model=model, criterion=criterion,
    optimizer=optimizer, epochs=epochs)
test_result = GddModelHandler().eval(test_loader, train_result.model, criterion)

model = train_result.model
model.to('cpu')
with open(os.path.join(log_dir, 'target_model.pkl'), 'wb') as f:
    save(model.state_dict(), f)

meta_data = LeakPro.make_mia_metadata(
    train_result=train_result, optimizer=optimizer, loss_fn=criterion,
    dataloader=train_loader, test_result=test_result, epochs=epochs,
    train_indices=train_indices, test_indices=test_indices, dataset_name=dataset_name)

with open(os.path.join(log_dir, 'model_metadata.pkl'), 'wb') as f:
    pickle.dump(meta_data, f)

print('init_params:', meta_data.init_params)
print(f'Target train acc {train_result.metrics.accuracy:.4f} | test acc {test_result.accuracy:.4f}')

## 4. Training curves

In [ ]:
import matplotlib.pyplot as plt

train_acc = train_result.metrics.extra['accuracy_history']
train_loss = train_result.metrics.extra['loss_history']

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.plot(train_acc, label='Train Accuracy')
plt.plot(len(train_acc) - 1, test_result.accuracy, 'ro', label='Test Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Accuracy'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(train_loss, label='Train Loss')
plt.plot(len(train_loss) - 1, test_result.loss, 'ro', label='Test Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss'); plt.legend()
plt.tight_layout(); plt.show()

## 5. Run the audit

Wires the split handlers: `GddDataHandler` (data) + `GddModelHandler` (model).

In [ ]:
config_path = 'audit.yaml'
from leakpro import LeakPro
from gdd_data_handler import GddDataHandler
from gdd_model_handler import GddModelHandler

leakpro = LeakPro(GddDataHandler, config_path, model_handler=GddModelHandler)
mia_results = leakpro.run_audit(create_pdf=True)